In [ ]:
!pip install transformers torch -q



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    pad_token_id=tokenizer.eos_token_id
).to(device)

print("✅ GPT-2 Model loaded successfully!")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ GPT-2 Model loaded successfully!


In [ ]:
# Writing a custom dataset to a text file
text_data = """
Artificial intelligence is revolutionizing the way we live and work.
Machine learning models are being used in healthcare to detect diseases early.
Natural language processing helps computers understand human language.
Deep learning has enabled breakthroughs in image and speech recognition.
AI-powered chatbots are now handling customer service across many industries.
The future of AI holds immense potential for solving global problems.
Neural networks are inspired by the structure of the human brain.
Data is the fuel that powers modern artificial intelligence systems.
Reinforcement learning allows agents to learn through trial and error.
Generative AI can create text, images, music, and even code.
"""

with open("dataset.txt", "w") as f:
    f.write(text_data)

print("✅ Dataset saved as dataset.txt")

✅ Dataset saved as dataset.txt


In [ ]:
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments
import torch

# Load dataset
class CustomTextDataset(torch.utils.data.Dataset):
    def __init__(self, file_path, tokenizer, block_size):
        with open(file_path, encoding="utf-8") as f:
            lines = [line for line in f.read().splitlines() if (len(line) > 0 and not line.isspace())]

        all_token_ids = []
        for line in lines:
            all_token_ids.extend(tokenizer.encode(line, add_special_tokens=True))

        self.examples = []
        for i in range(0, len(all_token_ids) - block_size + 1, block_size):
            self.examples.append(all_token_ids[i : i + block_size])

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return torch.tensor(self.examples[i])

def load_dataset(file_path, tokenizer, block_size=64):
    return CustomTextDataset(file_path, tokenizer, block_size)

# Data collator
def load_data_collator(tokenizer):
    return DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )

train_dataset = load_dataset("dataset.txt", tokenizer)
data_collator = load_data_collator(tokenizer)

# Training config
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    save_steps=100,
    logging_steps=10,
    save_total_limit=2,
    prediction_loss_only=True,
)

# Train!
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

print("🚀 Starting fine-tuning...")
trainer.train()
trainer.save_model("./gpt2-finetuned")
print("✅ Fine-tuning complete! Model saved.")

🚀 Starting fine-tuning...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,1.735776


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-tuning complete! Model saved.


In [ ]:
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments
import torch

# Load dataset
class CustomTextDataset(torch.utils.data.Dataset):
    def __init__(self, file_path, tokenizer, block_size):
        with open(file_path, encoding="utf-8") as f:
            lines = [line for line in f.read().splitlines() if (len(line) > 0 and not line.isspace())]

        all_token_ids = []
        for line in lines:
            all_token_ids.extend(tokenizer.encode(line, add_special_tokens=True))

        self.examples = []
        for i in range(0, len(all_token_ids) - block_size + 1, block_size):
            self.examples.append(all_token_ids[i : i + block_size])

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return torch.tensor(self.examples[i])

def load_dataset(file_path, tokenizer, block_size=64):
    return CustomTextDataset(file_path, tokenizer, block_size)

# Data collator
def load_data_collator(tokenizer):
    return DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )

train_dataset = load_dataset("dataset.txt", tokenizer)
data_collator = load_data_collator(tokenizer)

# Training config
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    save_steps=100,
    logging_steps=10,
    save_total_limit=2,
    prediction_loss_only=True,
)

# Train!
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

print("🚀 Starting fine-tuning...")
trainer.train()
trainer.save_model("./gpt2-finetuned")
print("✅ Fine-tuning complete! Model saved.")

🚀 Starting fine-tuning...


Step,Training Loss
10,0.291662


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-tuning complete! Model saved.


In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="./gpt2-finetuned")

prompts = [
    "Artificial intelligence is",
    "The future of machine learning",
    "Deep learning models can"
]

for prompt in prompts:
    result = generator(
        prompt,
        max_length=100,
        do_sample=True,
        top_p=0.92,
        top_k=50,
        temperature=0.85,
        num_return_sequences=1
    )
    print(f"\n📌 Prompt: '{prompt}'")
    print(result[0]["generated_text"])
    print("-" * 60)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_length', 'top_p', 'num_return_sequences', 'temperature', 'top_k'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📌 Prompt: 'Artificial intelligence is'
Artificial intelligence is revolutionizing the way we live and work.Machine learning models are being used in healthcare to detect diseases early.Natural language processing helps computers understand human language.Deep learning has enabled breakthroughs in image and speech recognition.AI-powered chatbots are now handling customer service across many industries.The future of finance is at stake with AI-powered financial services.The future of healthcare is at stake with advances in diagnostic imaging and early diagnosis.The future of healthcare is at stake with breakthroughs in cancer treatment.The future of healthcare is at stake with breakthroughs in heart disease treatment.The future of healthcare is at stake with breakthroughs in Alzheimer's disease and stroke treatment.The future of healthcare is at stake with breakthroughs in cancer treatment.The future of healthcare is at stake with breakthroughs in drug discovery and development.The futu

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📌 Prompt: 'The future of machine learning'
The future of machine learning is revolutionizing the way we live and work.Machine learning models are being used in healthcare to detect diseases early.Natural language processing helps computers understand human language.Deep learning has enabled breakthroughs in image and speech recognition.AI-powered chatbots are now handling customer service across many industries.The future of finance could revolutionise the way we live and work.With more than 2.5 billion customers globally, we deliver leading services to deliver value to customers.Machine learning models are being used in healthcare to detect diseases early.Natural language processing helps computers understand human language.Deep learning has enabled breakthroughs in image and speech recognition.AI-powered chatbots are now handling customer service across many industries.The future of finance could revolutionise the way we live and work.With more than 2.5 billion customers globally, w

In [ ]:
input_text = "Artificial intelligence is"
input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)

# Greedy
greedy = model.generate(input_ids, max_length=80)
print("🔹 GREEDY SEARCH:")
print(tokenizer.decode(greedy[0], skip_special_tokens=True))

# Beam Search
beam = model.generate(input_ids, max_length=80, num_beams=5, early_stopping=True)
print("\n🔹 BEAM SEARCH:")
print(tokenizer.decode(beam[0], skip_special_tokens=True))

# Top-P Sampling
topp = model.generate(input_ids, max_length=80, do_sample=True, top_p=0.92, top_k=0, temperature=0.9)
print("\n🔹 TOP-P SAMPLING:")
print(tokenizer.decode(topp[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


🔹 GREEDY SEARCH:
Artificial intelligence is revolutionizing the way we live and work.Machine learning models are being used in healthcare to detect diseases early.Natural language processing helps computers understand human language.Deep learning has enabled breakthroughs in image and speech recognition.AI-powered chatbots are now handling customer service across many industries.The future of finance is in the digital age of trustless lending.The future of health and


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🔹 BEAM SEARCH:
Artificial intelligence is revolutionizing the way we live and work.Machine learning models are being used in healthcare to detect diseases early.Natural language processing helps computers understand human language.Deep learning has enabled breakthroughs in image and speech recognition.Deep learning has enabled breakthroughs in image and speech recognition.AI-powered chatbots are now handling customer service across many industries.AI-powered chatbots are now

🔹 TOP-P SAMPLING:
Artificial intelligence is revolutionizing the way we live and work.Machine learning models are being used in healthcare to detect diseases early.Natural language processing helps computers understand human language.Deep learning has allowed breakthroughs in image and speech recognition.AI-powered chatbots are now handling customer service across many industries.Our team includes world-class talent and passionate people.With more than 1.4 million users
